# Data Preprocessing

In [3]:
import os
from collections import defaultdict
import json
import torchaudio
import torchaudio.functional as F
# For class specific prompts
# from tinytag import TinyTag

In [5]:
input_root = "./data"
output_root = "./data_preprocessed"
target_sr = 44100
target_channels = 2

class_mapping = defaultdict(int)
dropped = 0

for root, dirs, files in os.walk(input_root):
    head, class_name = os.path.split(root)

    # skips non class directories
    if not files:
        continue

    # xeno canto has one extra class not contained in kaggle, so it may not be perfectly alphabetically
    if class_name not in class_mapping:
        class_mapping[class_name] = len(class_mapping)

    # create output dir for each class
    output_path = os.path.join(output_root, class_name)
    os.makedirs(output_path, exist_ok=True)

    for file in files:
        if file.lower().endswith(".mp3"):
            # define paths
            in_file = os.path.join(root, file)
            filename = os.path.splitext(file)[0]
            out_file = os.path.join(output_path, filename)
            try:
                waveform, sr = torchaudio.load(in_file)
                # resample
                if sr != target_sr:
                    waveform = F.resample(waveform, sr, target_sr)

                # convert channels
                num_channels = waveform.shape[0]
                if num_channels == 1:
                    waveform = waveform.repeat(target_channels, 1)
                elif num_channels > 2:
                    waveform = waveform[:2, :]

                # convert to wav
                torchaudio.save(out_file + ".wav", waveform, target_sr)

                # add json containing "prompt" and IntCondition
                cur_class_id = class_mapping[class_name]
                
                # For class specific prompt
                # tag = TinyTag.get(in_file)
                # bird_title = tag.title if tag.title else class_name
                # prompt = f"Field recording of a {bird_title}."
                
                # general prompt
                prompt = "A field recording of a bird singing in nature, stereo audio."
                
                conditioning_dict = {
                    "prompt": prompt,
                    "species_id": cur_class_id,
                }
                json.dump(conditioning_dict, open(out_file + ".json", "w"))

            except Exception as e:
                dropped += 1
                print(f"Failed to process {in_file}, error: {e}")

with open("class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=4)
print("Class mappings saved")
print("Files processed successfully. Samples dropped:", dropped)

Failed to process ./data\xenoCanto\amepip\XC113723.mp3, error: Unspecified internal error.
Failed to process ./data\xenoCanto\dusfly\XC356012.mp3, error: Unspecified internal error.
Class mappings saved
Files processed successfully. Samples dropped: 2
